# Patient Fixed-Effects Robustness Analysis

Purpose: evaluate whether the principal encounter-adjusted associations are robust to conditioning on patient-specific fixed effects among repeated HCW-initiated threads. This is a robustness analysis and does not replace the primary pooled logistic regression models.

The notebook compares 30-day and 90-day exposure specifications using patient reply within 7 days as the outcome. It reports the effective sample after fixed-effects conditioning, compares principal exposure coefficients with the primary encounter-adjusted model, and saves aggregate model and sample summaries only. Row-level EHR-derived data are not exported.


## 1. Estimation Approach

Patient fixed effects are estimated with conditional logistic regression (`statsmodels.discrete.conditional_models.ConditionalLogit`). The conditional likelihood removes the patient-specific intercept without creating one indicator variable per patient.

Only patients with within-patient variation in the binary reply outcome contribute to the conditional likelihood. Patients whose eligible threads all have the same outcome are excluded by design. Consequently, the fixed-effects estimand applies to patients with both replied and nonreplied threads during the analysis period.

Time-invariant patient characteristics such as sex, race, and ethnicity are absorbed by patient fixed effects and are not separately estimated. Age at message is omitted from the conditional model because patient fixed effects together with month-year fixed effects absorb its identifying variation. Time-varying Charlson group and month-year fixed effects are retained when they have within-patient variation.

For comparison, this notebook fits: (1) the primary encounter-adjusted pooled model on its complete-case sample, (2) the same pooled model on the fixed-effects-conditioned sample, and (3) the conditional patient fixed-effects model.


In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.metadata as importlib_metadata
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import patsy
import statsmodels
import statsmodels.formula.api as smf
from scipy.stats import norm
from statsmodels.discrete.conditional_models import ConditionalLogit
from IPython.display import display

pd.options.display.max_columns = 160
pd.options.display.float_format = "{:.4f}".format


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORKING_DATA_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()
INPUT_PATH = WORKING_DATA_DIR / "analysis_df_all_exposures_with_encounter_covariates.parquet"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / "runs" / f"patient_fixed_effects_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing unified analysis dataset: {INPUT_PATH}\n"
        "Run 05_build_exposure_analysis_dataset_clean.ipynb first."
    )

OUTCOME = "patient_replied_7d"
PERIODS = ("30d", "90d")
ENCOUNTER_CONTROL = "log_encounters_past_7d"
ANALYTIC_START = pd.Timestamp("2021-03-31")
ANALYTIC_END = pd.Timestamp("2023-11-30 23:59:59.999999999")
MAXITER = 200
SIGNIFICANCE_LEVEL = 0.05

print("Estimator: statsmodels ConditionalLogit")
print("statsmodels version:", statsmodels.__version__)
print("INPUT_PATH:", INPUT_PATH)
print("RUN_DIR:", RUN_DIR)

## 2. Load And Prepare Analysis Data

The preparation below verifies the manuscript analytic restriction applied upstream and mirrors the primary categorical coding. No patient or message-level examples are displayed.


In [ ]:
analysis_df = pd.read_parquet(INPUT_PATH, engine="pyarrow")

required_columns = [
    "thread_id", "DMRN", "thread_first_time", OUTCOME,
    "Age65", "Age_at_Message", "Gender", "Race",
    "Ethnicity", "Charlson_group", ENCOUNTER_CONTROL,
]
for period in PERIODS:
    required_columns.extend([
        f"log_provider_msgs_past_{period}",
        f"log_system_msgs_past_{period}",
        f"log_patient_initiated_threads_past_{period}",
    ])
missing_columns = sorted(set(required_columns) - set(analysis_df.columns))
if missing_columns:
    raise KeyError("Missing required analysis columns: " + ", ".join(missing_columns))

if analysis_df["thread_id"].duplicated().any():
    raise ValueError("The analysis dataset must contain one row per thread_id.")

analysis_df["Age65"] = pd.Categorical(
    analysis_df["Age65"], categories=["<65", "\u226565"], ordered=False
)
analysis_df["Ethnicity"] = pd.Categorical(
    analysis_df["Ethnicity"],
    categories=["Not Hispanic/Latino", "Hispanic/Latino", "Unknown/Other"],
    ordered=False,
)
analysis_df["Charlson_group"] = analysis_df["Charlson_group"].astype("category")
analysis_df["thread_first_time"] = pd.to_datetime(analysis_df["thread_first_time"], errors="coerce")
analysis_df["month_year"] = analysis_df["thread_first_time"].dt.to_period("M").astype(str)

analysis_df["Race"] = np.select(
    [
        analysis_df["Race"].eq("White"),
        analysis_df["Race"].eq("Black/African American"),
        analysis_df["Race"].eq("Asian"),
    ],
    ["White", "Black/African American", "Asian"],
    default="Other/Multiple/Unknown",
)
analysis_df["Race"] = pd.Categorical(
    analysis_df["Race"],
    categories=["White", "Black/African American", "Asian", "Other/Multiple/Unknown"],
    ordered=False,
)

analysis_df[OUTCOME] = analysis_df[OUTCOME].astype(float).fillna(0).astype(int)
start_cutoff = ANALYTIC_START
end_cutoff = ANALYTIC_END
if analysis_df["Age65"].isna().any():
    raise ValueError("Unified analysis dataset contains missing Age65 values. Rerun notebook 05.")
if not analysis_df["thread_first_time"].between(start_cutoff, end_cutoff, inclusive="both").all():
    raise ValueError("Unified analysis dataset contains rows outside the manuscript analytic period. Rerun notebook 05.")
filtered_df = analysis_df.copy()

print("Rows loaded:", f"{len(analysis_df):,}")
print("Rows in canonical manuscript sample:", f"{len(filtered_df):,}")
print("Patients in canonical manuscript sample:", f"{filtered_df['DMRN'].nunique():,}")


## 3. Construct Complete-Case And Conditional Samples

The pooled and conditional models are compared on an identical fixed-effects-conditioned sample. The full primary pooled model is also retained as the reference specification.


In [ ]:
def construct_period_samples(df: pd.DataFrame, period: str):
    x_provider = f"log_provider_msgs_past_{period}"
    x_system = f"log_system_msgs_past_{period}"
    x_patient_threads = f"log_patient_initiated_threads_past_{period}"

    complete_case_columns = [
        OUTCOME, "DMRN", "month_year", "Age_at_Message",
        "Gender", "Race", "Ethnicity", "Charlson_group",
        ENCOUNTER_CONTROL, x_provider, x_system, x_patient_threads,
    ]
    complete = df.dropna(subset=complete_case_columns).copy()

    patient_outcomes = (
        complete.groupby("DMRN", observed=True)[OUTCOME]
        .agg(threads="size", replies="sum", outcome_levels="nunique")
    )
    informative_ids = patient_outcomes.index[patient_outcomes["outcome_levels"].eq(2)]
    conditioned = complete.loc[complete["DMRN"].isin(informative_ids)].copy()
    conditioned = conditioned.sort_values(["DMRN", "thread_first_time", "thread_id"], kind="mergesort")

    no_reply = patient_outcomes["replies"].eq(0)
    all_reply = patient_outcomes["replies"].eq(patient_outcomes["threads"])
    summary = {
        "period": period,
        "canonical_rows_loaded": int(len(df)),
        "complete_case_rows": int(len(complete)),
        "complete_case_patients": int(complete["DMRN"].nunique()),
        "patients_all_no_reply": int(no_reply.sum()),
        "patients_all_reply": int(all_reply.sum()),
        "patients_excluded_no_outcome_variation": int((patient_outcomes["outcome_levels"] == 1).sum()),
        "rows_excluded_no_outcome_variation": int(len(complete) - len(conditioned)),
        "conditional_rows": int(len(conditioned)),
        "conditional_patients": int(conditioned["DMRN"].nunique()),
        "conditional_row_retention_pct": float(len(conditioned) / len(complete) * 100) if len(complete) else np.nan,
    }
    return complete, conditioned, summary


## 4. Fit Pooled And Conditional Logistic Models

The pooled models use patient-clustered standard errors. The conditional model uses patient-level conditional likelihood and a patient-group sandwich covariance calculated from conditional score contributions.


In [ ]:
def primary_formula(period: str) -> str:
    x_provider = f"log_provider_msgs_past_{period}"
    x_system = f"log_system_msgs_past_{period}"
    x_patient_threads = f"log_patient_initiated_threads_past_{period}"
    terms = [
        x_provider, x_system, f"{x_provider}:{x_system}",
        x_patient_threads, ENCOUNTER_CONTROL, "Age_at_Message",
        "C(Gender)", 'C(Race, Treatment(reference="White"))',
        "Ethnicity", "C(Charlson_group)", "C(month_year)",
    ]
    return f"{OUTCOME} ~ " + " + ".join(terms)


def fit_pooled_model(df: pd.DataFrame, period: str):
    return smf.logit(primary_formula(period), data=df).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["DMRN"]},
        disp=False,
    )


def build_conditional_design(df: pd.DataFrame, period: str):
    x_provider = f"log_provider_msgs_past_{period}"
    x_system = f"log_system_msgs_past_{period}"
    x_patient_threads = f"log_patient_initiated_threads_past_{period}"
    interaction = f"provider_system_interaction_{period}"

    work = df.copy()
    work[interaction] = work[x_provider] * work[x_system]
    rhs = " + ".join([
        x_provider, x_system, interaction, x_patient_threads,
        ENCOUNTER_CONTROL, "C(Charlson_group)", "C(month_year)",
    ])
    design = patsy.dmatrix(rhs, data=work, return_type="dataframe")
    design = design.drop(columns=["Intercept"], errors="ignore")

    groups = work.loc[design.index, "DMRN"]
    group_means = design.groupby(groups, observed=True).transform("mean")
    within_range = (design - group_means).abs().max(axis=0)
    no_within_variation = within_range.index[within_range.le(1e-12)].tolist()
    design = design.drop(columns=no_within_variation)

    principal_terms = [x_provider, x_system, interaction, x_patient_threads, ENCOUNTER_CONTROL]
    missing_principal_terms = [term for term in principal_terms if term not in design.columns]
    if missing_principal_terms:
        raise ValueError(
            "Principal terms have no within-patient variation: "
            + ", ".join(missing_principal_terms)
        )

    return work.loc[design.index].copy(), design, no_within_variation, interaction


def conditional_group_robust_covariance(model, params):
    params_array = np.asarray(params, dtype=float)
    scores = np.vstack([
        np.asarray(model.score_grp(group_index, params_array), dtype=float)
        for group_index in range(model._n_groups)
    ])
    bread = np.linalg.pinv(-np.asarray(model.hessian(params_array), dtype=float))
    meat = scores.T @ scores
    covariance = bread @ meat @ bread
    if model._n_groups > 1:
        covariance *= model._n_groups / (model._n_groups - 1)
    return covariance


In [ ]:
model_results = {}
sample_rows = []
diagnostic_rows = []

for period in PERIODS:
    print(f"\nPreparing {period} models")
    complete_sample, conditional_sample, sample_summary = construct_period_samples(filtered_df, period)
    sample_rows.append(sample_summary)
    display(pd.DataFrame([sample_summary]))

    print("Fitting primary pooled model on the full complete-case sample")
    pooled_full = fit_pooled_model(complete_sample, period)

    print("Fitting pooled model on the fixed-effects-conditioned sample")
    pooled_conditioned = fit_pooled_model(conditional_sample, period)

    fe_data, fe_design, dropped_terms, interaction_term = build_conditional_design(
        conditional_sample, period
    )
    print("Fitting conditional patient fixed-effects model")
    conditional_model = ConditionalLogit(
        fe_data[OUTCOME],
        fe_design,
        groups=fe_data["DMRN"],
    )
    conditional_result = conditional_model.fit(
        method="BFGS", maxiter=MAXITER, disp=False
    )
    conditional_cov = conditional_group_robust_covariance(
        conditional_model, conditional_result.params
    )

    conditional_score_max_abs = float(
        np.max(np.abs(conditional_model.score(conditional_result.params)))
    )
    converged = conditional_score_max_abs < 1e-4
    diagnostic_rows.append({
        "period": period,
        "conditional_rows": int(conditional_result.nobs),
        "conditional_patients": int(conditional_result.n_groups),
        "design_columns": int(fe_design.shape[1]),
        "terms_dropped_no_within_variation": "; ".join(dropped_terms),
        "conditional_score_max_abs": conditional_score_max_abs,
        "converged": converged,
    })
    model_results[period] = {
        "pooled_full": pooled_full,
        "pooled_conditioned": pooled_conditioned,
        "conditional_result": conditional_result,
        "conditional_cov": conditional_cov,
        "conditional_design_columns": list(fe_design.columns),
        "interaction_term": interaction_term,
    }

sample_summary = pd.DataFrame(sample_rows)
model_diagnostics = pd.DataFrame(diagnostic_rows)
display(sample_summary)
display(model_diagnostics)


## 5. Compare Principal Exposure Coefficients

The provider and system main effects are interpreted at zero on the other interacting exposure. The comparison table reports odds ratios, confidence intervals, P values, direction agreement, and whether statistical-significance classifications agree with the primary model.


In [ ]:
def pooled_term_statistics(model, term: str):
    beta = float(model.params[term])
    interval = model.conf_int().loc[term].astype(float)
    return {
        "beta": beta,
        "OR": float(np.exp(beta)),
        "CI_low": float(np.exp(interval.iloc[0])),
        "CI_high": float(np.exp(interval.iloc[1])),
        "P_value": float(model.pvalues[term]),
    }


def conditional_term_statistics(result, covariance, columns, term: str):
    params = pd.Series(np.asarray(result.params, dtype=float), index=columns)
    standard_errors = pd.Series(np.sqrt(np.diag(covariance)), index=columns)
    beta = float(params[term])
    standard_error = float(standard_errors[term])
    z_value = beta / standard_error
    p_value = float(2 * norm.sf(abs(z_value)))
    return {
        "beta": beta,
        "OR": float(np.exp(beta)),
        "CI_low": float(np.exp(beta - 1.96 * standard_error)),
        "CI_high": float(np.exp(beta + 1.96 * standard_error)),
        "P_value": p_value,
    }


comparison_rows = []
for period, objects in model_results.items():
    x_provider = f"log_provider_msgs_past_{period}"
    x_system = f"log_system_msgs_past_{period}"
    pooled_interaction = f"{x_provider}:{x_system}"
    x_patient_threads = f"log_patient_initiated_threads_past_{period}"
    fe_interaction = objects["interaction_term"]
    term_map = [
        ("Provider messages", "principal exposure", x_provider, x_provider),
        ("System messages", "principal exposure", x_system, x_system),
        ("Provider x system interaction", "principal exposure", pooled_interaction, fe_interaction),
        ("Patient-initiated threads", "principal exposure", x_patient_threads, x_patient_threads),
        ("Encounters, past 7 days", "control", ENCOUNTER_CONTROL, ENCOUNTER_CONTROL),
    ]

    for label, role, pooled_term, fe_term in term_map:
        primary = pooled_term_statistics(objects["pooled_full"], pooled_term)
        pooled_same = pooled_term_statistics(objects["pooled_conditioned"], pooled_term)
        patient_fe = conditional_term_statistics(
            objects["conditional_result"],
            objects["conditional_cov"],
            objects["conditional_design_columns"],
            fe_term,
        )
        comparison_rows.append({
            "period": period,
            "variable": label,
            "role": role,
            "primary_beta": primary["beta"],
            "primary_OR": primary["OR"],
            "primary_CI_low": primary["CI_low"],
            "primary_CI_high": primary["CI_high"],
            "primary_P_value": primary["P_value"],
            "pooled_conditioned_OR": pooled_same["OR"],
            "pooled_conditioned_P_value": pooled_same["P_value"],
            "patient_FE_beta": patient_fe["beta"],
            "patient_FE_OR": patient_fe["OR"],
            "patient_FE_CI_low": patient_fe["CI_low"],
            "patient_FE_CI_high": patient_fe["CI_high"],
            "patient_FE_P_value": patient_fe["P_value"],
        })

coefficient_comparison = pd.DataFrame(comparison_rows)
coefficient_comparison["direction_same_as_primary"] = (
    np.sign(coefficient_comparison["primary_beta"])
    == np.sign(coefficient_comparison["patient_FE_beta"])
)
coefficient_comparison["significance_same_as_primary"] = (
    coefficient_comparison["primary_P_value"].lt(SIGNIFICANCE_LEVEL)
    == coefficient_comparison["patient_FE_P_value"].lt(SIGNIFICANCE_LEVEL)
)
coefficient_comparison["patient_FE_vs_primary_beta_difference"] = (
    coefficient_comparison["patient_FE_beta"]
    - coefficient_comparison["primary_beta"]
)

display(coefficient_comparison)


## 6. Aggregate Robustness Summary And Outputs


In [ ]:
principal_comparison = coefficient_comparison.loc[
    coefficient_comparison["role"].eq("principal exposure")
].copy()

direction_stable = bool(principal_comparison["direction_same_as_primary"].all())
significance_stable = bool(principal_comparison["significance_same_as_primary"].all())
if direction_stable and significance_stable:
    overall_summary = "All principal exposure directions and statistical-significance classifications were unchanged."
elif direction_stable:
    overall_summary = "All principal exposure directions were unchanged, but at least one statistical-significance classification differed."
else:
    overall_summary = "At least one principal exposure changed direction in the patient fixed-effects analysis."

robustness_summary = pd.DataFrame([{
    "principal_terms_compared": int(len(principal_comparison)),
    "principal_terms_same_direction": int(principal_comparison["direction_same_as_primary"].sum()),
    "principal_terms_same_significance_classification": int(principal_comparison["significance_same_as_primary"].sum()),
    "all_principal_directions_stable": direction_stable,
    "all_principal_significance_classifications_stable": significance_stable,
    "summary": overall_summary,
}])
display(robustness_summary)
print(overall_summary)

sample_path = RUN_DIR / "patient_fe_effective_sample.csv"
diagnostics_path = RUN_DIR / "patient_fe_model_diagnostics.csv"
comparison_path = RUN_DIR / "patient_fe_principal_coefficient_comparison.csv"
summary_path = RUN_DIR / "patient_fe_robustness_summary.csv"
metadata_path = RUN_DIR / "patient_fe_reproducibility_summary.json"

sample_summary.to_csv(sample_path, index=False)
model_diagnostics.to_csv(diagnostics_path, index=False)
coefficient_comparison.to_csv(comparison_path, index=False)
robustness_summary.to_csv(summary_path, index=False)

metadata = {
    "run_id": RUN_ID,
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "input_file": str(INPUT_PATH),
    "estimator": "statsmodels ConditionalLogit with patient groups",
    "outcome": OUTCOME,
    "periods": list(PERIODS),
    "encounter_control": ENCOUNTER_CONTROL,
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "patsy": importlib_metadata.version("patsy"),
        "scipy": importlib_metadata.version("scipy"),
        "statsmodels": statsmodels.__version__,
    },
    "privacy_restriction": (
        "Row-level EHR-derived records are restricted. Only aggregate sample and model summaries are saved."
    ),
}
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved aggregate outputs:")
for path in [sample_path, diagnostics_path, comparison_path, summary_path, metadata_path]:
    print(" -", path)


## Generate Table S7

In [ ]:
# Table S7. Encounter-adjusted pooled vs patient fixed-effects
# ============================================================

from IPython.display import display, Markdown
import numpy as np
import pandas as pd


def format_or_ci(or_value, ci_low, ci_high, decimals=2):
    """Format an odds ratio and 95% confidence interval."""
    values = [or_value, ci_low, ci_high]
    if not all(np.isfinite(values)):
        return ""
    return (
        f"{or_value:.{decimals}f} "
        f"({ci_low:.{decimals}f}-{ci_high:.{decimals}f})"
    )


table_s7_rows = []

for period in ("30d", "90d"):
    objects = model_results[period]

    hcw_term = f"log_provider_msgs_past_{period}"
    system_term = f"log_system_msgs_past_{period}"
    pooled_interaction_term = f"{hcw_term}:{system_term}"
    fe_interaction_term = objects["interaction_term"]
    patient_messaging_term = f"log_patient_initiated_threads_past_{period}"

    terms = [
        ("HCW messages", hcw_term, hcw_term),
        ("System messages", system_term, system_term),
        (
            "HCW x System interaction",
            pooled_interaction_term,
            fe_interaction_term,
        ),
        (
            "Patient-initiated messaging",
            patient_messaging_term,
            patient_messaging_term,
        ),
        (
            "Recent encounters (log1p, past 7 days)",
            ENCOUNTER_CONTROL,
            ENCOUNTER_CONTROL,
        ),
    ]

    for row_number, (label, pooled_term, fe_term) in enumerate(terms):
        # Primary encounter-adjusted pooled model
        pooled_stats = pooled_term_statistics(
            objects["pooled_full"],
            pooled_term,
        )

        # Patient fixed-effects conditional logistic regression
        fe_stats = conditional_term_statistics(
            objects["conditional_result"],
            objects["conditional_cov"],
            objects["conditional_design_columns"],
            fe_term,
        )

        table_s7_rows.append(
            {
                "Message exposure window": (
                    "30 days" if period == "30d" else "90 days"
                ) if row_number == 0 else "",
                "Variable": label,
                "Encounter-adjusted pooled specification OR (95% CI)": (
                    format_or_ci(
                        pooled_stats["OR"],
                        pooled_stats["CI_low"],
                        pooled_stats["CI_high"],
                    )
                ),
                "Patient fixed-effects specification OR (95% CI)": (
                    format_or_ci(
                        fe_stats["OR"],
                        fe_stats["CI_low"],
                        fe_stats["CI_high"],
                    )
                ),
            }
        )


table_s7 = pd.DataFrame(table_s7_rows)

table_s7_title = (
    "Table S7. Comparison of the encounter-adjusted pooled and "
    "patient fixed-effects model specifications."
)

table_s7_note = (
    "Note. Both models were estimated on the same subset of HCW-initiated threads with complete encounter information to permit a direct comparison between the primary and encounter-adjusted specifications."
    "The encounter-adjusted specification additionally included the log-transformed number of patient encounters during the 7 days preceding thread initiation (log[1 + encounters])."
    "Only principal exposure variables are shown; demographic covariates, comorbidity measures, and calendar fixed effects are omitted for brevity."
    "Odds ratios are presented to facilitate direct comparison of the principal exposure effects before and after encounter adjustment."
)

display(Markdown(f"**{table_s7_title}**"))
display(table_s7)
display(Markdown(table_s7_note))


# ------------------------------------------------------------
# Save the formatted table and note
# ------------------------------------------------------------

table_s7_path = RUN_DIR / "Table_S7_patient_fixed_effects_comparison.csv"
table_s7_note_path = RUN_DIR / "Table_S7_patient_fixed_effects_note.txt"

table_s7.to_csv(
    table_s7_path,
    index=False,
    encoding="utf-8-sig",
)

table_s7_note_path.write_text(
    table_s7_title + "\n\n" + table_s7_note,
    encoding="utf-8",
)

print("Saved Table S7:")
print(" -", table_s7_path)
print(" -", table_s7_note_path)

## Interpretation Notes

The conditional estimates are identified only by within-patient contrasts among patients who have both replied and nonreplied eligible threads. Differences from the primary model may therefore reflect both control for time-invariant patient characteristics and a change from the full population to the informative-patient sample. The pooled model fitted on the conditioned sample helps distinguish these two mechanisms.

The robustness analysis should be interpreted alongside the primary pooled models and not as a replacement estimand.
